In [ ]:
from sklearn.datasets import fetch_lfw_people
lfw_people = fetch_lfw_people(min_faces_per_person=100)
print(lfw_people.images.shape, lfw_people.data.shape)
print(lfw_people.target_names)

In [ ]:
## ##

import numpy as np

np.min(lfw_people.images), np.max(lfw_people.images)

# = images는 0 ~ 1의 구간값을 가지고 있음 = normalize 되어 있음

In [ ]:
## target값이 몇 개씩 있는지 확인 ##

np.unique(lfw_people.target, return_counts = True)

# = target에 0 ~ 4가 nnn개씩 있음
# = 분류를 위해 target값 원핫인코딩 필요

In [ ]:
## target값 원핫인코딩 ##

from tensorflow.keras.utils import to_categorical
y_data = to_categorical(lfw_people.target)
print(y_data.shape)
print(y_data)

In [ ]:
## 모델 생성 ##
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Conv2D, MaxPool2D, Flatten, Dense

lfw_model = Sequential()
lfw_model.summary()

# layer를 아무것도 안 넣고 summary를 하니 빈 테이블이 나옴

In [ ]:
## 적절한 레이어 추가 ##
lfw_model = Sequential()

lfw_model.add(Input(shape = (62, 47, 1)))    # 세로크기, 가로크기, 채널 수

lfw_model.add(Conv2D(32, kernel_size=(3,3), activation='relu'))

lfw_model.summary()

# Param # 320 = 필터 사이즈 3 * 3
# 필터링 Output Shape가 줄어든 이유 : zero_padding을 안해서
# 패딩을 하는 이유 - 이미지가 저리 걸린 순간   / 사실 지금의 데이터셋에는 패딩이 필요하지  않지만,,

In [ ]:
## 적절한 레이어 추가 - padding 추가 ##
lfw_model = Sequential()

lfw_model.add(Input(shape = (62, 47, 1)))    # 세로크기, 가로크기, 채널 수

lfw_model.add(Conv2D(32, kernel_size=(3,3), padding = 'same', activation='relu'))

lfw_model.summary()

In [ ]:
## 컴파일 + 학습 ##
lfw_model = Sequential()

lfw_model.add(Input(shape = (62, 47, 1)))    # 세로크기, 가로크기, 채널 수

lfw_model.add(Conv2D(32, kernel_size=(3,3), padding = 'same', activation='relu'))
lfw_model.add(Flatten())      # shape 1차원으로

lfw_model.add(Dense(5, activation= 'softmax'))

lfw_model.summary()
lfw_model.compile(loss = 'categorical_crossentropy', optimizer = 'adam',
                  metrics = ['accuracy'])

lfw_model.fit(lfw_people.images, y_data ,
              validation_split= 0.3, epochs = 1)

In [ ]:
x_data = lfw_people.images[:,:,:,np.newaxis]
print(x_data.shape)

In [ ]:
## MaxPool2D 추가 - 도드라지는 특성 뽑기

lfw_model = Sequential()

lfw_model.add(Input(shape=(62,47,1)))

lfw_model.add(Conv2D(32, kernel_size=(3,3), padding='same', activation='relu'))
lfw_model.add(MaxPool2D(pool_size=(2,2)))                          # 2 * 2 영역 중 제일 도드라지는 특성 뽑기 (최대값)
lfw_model.add(Conv2D(32, kernel_size=(3,3), activation='relu'))    # 패딩 생략 -> Output shape 줄어듦
lfw_model.add(MaxPool2D(pool_size=(2,2)))
lfw_model.add(Flatten())

lfw_model.add(Dense(5, activation='softmax'))     # 클래스 수가 5개 -> 뉴런 5개 + softmax

lfw_model.summary()
lfw_model.compile(loss='categorical_crossentropy', optimizer='adam',
                  metrics=['accuracy'])

lfw_model.fit(lfw_people.images, y_data,
              validation_split=0.3,
              epochs=1)


# param# 9248 = 3 * 3 * 32 * 32 +32
#  22,405 - fully connected

In [ ]:
# * 참고
#    학습 속도는 Dense가 빠르지만, 실제 사용 시에는 Conv 구조가 더 효율적이고 가볍다.
#    Dense는 파라미터 수가 많아서 무겁고, 메모리 부담이 크다.

In [ ]:
## early_stopping 추가 + train/test 성능 시각화 ##

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Conv2D, MaxPool2D, Flatten, Dense
from tensorflow.keras.callbacks import EarlyStopping

lfw_model = Sequential()

lfw_model.add(Input(shape=(62,47,1)))

lfw_model.add(Conv2D(32, kernel_size=(3,3), padding='same', activation='relu'))
lfw_model.add(MaxPool2D(pool_size=(2,2)))
lfw_model.add(Conv2D(32, kernel_size=(3,3), activation='relu'))
lfw_model.add(MaxPool2D(pool_size=(2,2)))
lfw_model.add(Conv2D(32, kernel_size=(3,3), activation='relu'))
lfw_model.add(MaxPool2D(pool_size=(2,2)))
lfw_model.add(Flatten())
#lfw_model.add(Dense(128, activation='relu'))

lfw_model.add(Dense(5, activation='softmax'))

lfw_model.summary()
lfw_model.compile(loss='categorical_crossentropy', optimizer='adam',
                  metrics=['accuracy'])

esc = EarlyStopping(monitor='val_loss', patience=5)

hist = lfw_model.fit(lfw_people.images, y_data,
                     validation_split=0.3,
                     epochs=5, callbacks=[esc])


import matplotlib.pyplot as plt
plt.plot(range(1,len(hist.history['val_loss'])+1), hist.history['val_loss'], label='Test')
plt.plot(range(1,len(hist.history['loss'])+1), hist.history['loss'], label='Train')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.grid()
plt.legend()
plt.show()

In [ ]:
## Dense 레이어 추가 ##

lfw_model = Sequential()
lfw_model.add(Input(shape=(62,47,1)))

lfw_model.add(Conv2D(32, kernel_size=(3,3), padding='same', activation='relu'))
lfw_model.add(MaxPool2D(pool_size=(2,2)))
lfw_model.add(Conv2D(32, kernel_size=(3,3), activation='relu'))
lfw_model.add(MaxPool2D(pool_size=(2,2)))
lfw_model.add(Conv2D(32, kernel_size=(3,3), activation='relu'))
lfw_model.add(MaxPool2D(pool_size=(2,2)))
lfw_model.add(Flatten())

lfw_model.add(Dense(5, activation='softmax'))

lfw_model.summary()
lfw_model.compile(loss='categorical_crossentropy', optimizer='adam',
                  metrics=['accuracy'])

esc = EarlyStopping(monitor='val_loss', patience=5)

hist = lfw_model.fit(lfw_people.images, y_data,
                     validation_split=0.3,
                     epochs=500, callbacks=[esc])

import matplotlib.pyplot as plt
plt.plot(range(1,len(hist.history['val_loss'])+1), hist.history['val_loss'], label='Test')
plt.plot(range(1,len(hist.history['loss'])+1), hist.history['loss'], label='Train')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.grid()
plt.legend()
plt.show()


lfw_model = Sequential()
lfw_model.add(Input(shape=(62,47,1)))

lfw_model.add(Conv2D(32, kernel_size=(3,3), padding='same', activation='relu'))
lfw_model.add(MaxPool2D(pool_size=(2,2)))
lfw_model.add(Conv2D(32, kernel_size=(3,3), activation='relu'))
lfw_model.add(MaxPool2D(pool_size=(2,2)))
lfw_model.add(Flatten())
lfw_model.add(Dense(128, activation='relu'))

lfw_model.add(Dense(5, activation='softmax'))

lfw_model.summary()
lfw_model.compile(loss='categorical_crossentropy', optimizer='adam',
                  metrics=['accuracy'])

esc = EarlyStopping(monitor='val_loss', patience=5)

hist = lfw_model.fit(lfw_people.images, y_data,
                     validation_split=0.3,
                     epochs=500, callbacks=[esc])

import matplotlib.pyplot as plt
plt.plot(range(1,len(hist.history['val_loss'])+1), hist.history['val_loss'], label='Test')
plt.plot(range(1,len(hist.history['loss'])+1), hist.history['loss'], label='Train')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.grid()
plt.legend()
plt.show()

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
rootdir= '/content/drive/MyDrive/25_영등포_류은수/2025/2025_SeSAC/python_programming/한상훈 강사님_딥러닝/data/lfw_model.keras'

In [ ]:
## ModelCheckpoint : Best model 저장

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Conv2D, MaxPool2D, Flatten, Dense
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

lfw_model = Sequential()
lfw_model.add(Input(shape=(62,47,1)))

lfw_model.add(Conv2D(32, kernel_size=(3,3), padding='same', activation='relu'))
lfw_model.add(MaxPool2D(pool_size=(2,2)))
lfw_model.add(Conv2D(32, kernel_size=(3,3), activation='relu'))
lfw_model.add(MaxPool2D(pool_size=(2,2)))
lfw_model.add(Conv2D(32, kernel_size=(3,3), activation='relu'))
lfw_model.add(MaxPool2D(pool_size=(2,2)))
lfw_model.add(Flatten())

lfw_model.add(Dense(5, activation='softmax'))

lfw_model.summary()
lfw_model.compile(loss='categorical_crossentropy', optimizer='adam',
                  metrics=['accuracy'])

esc = EarlyStopping(monitor='val_loss', patience=5)

cpt = ModelCheckpoint(filepath=rootdir+'lfw_model.keras', monitor='val_loss',
                      save_best_only=True, verbose=1)

hist = lfw_model.fit(lfw_people.images, y_data,
                     validation_split=0.3,
                     epochs=500, callbacks=[esc, cpt])

import matplotlib.pyplot as plt
plt.plot(range(1,len(hist.history['val_loss'])+1), hist.history['val_loss'], label='Test')
plt.plot(range(1,len(hist.history['loss'])+1), hist.history['loss'], label='Train')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.grid()
plt.legend()
plt.show()

In [ ]:
# train, test의 성능을 정확도와 loss로 시각화

import matplotlib.pyplot as plt
def train_report(hist):
  fig, axs = plt.subplots(nrows=1, ncols=2, layout='tight', figsize=(7,3))
  for i, mt, in enumerate(['accuracy','val_accuracy','loss','val_loss']):
    if i%2 == 0:
      axs.flat[i//2].set_title(mt)
    axs.flat[i//2].plot(range(1,len(hist.history[mt])+1), hist.history[mt], label=mt)
  plt.grid()
  plt.legend()
  plt.show()

train_report(hist)

In [ ]:
##
from tensorflow.keras.models import load_model
lfw_model.summary()
del(lfw_model)    # 메모리 내 해당 모델 제거
lfw_model.summary()

In [ ]:
## 우리가 저장한 best model 불러오기
lfw_model = load_model(rootdir + 'lfw_model.keras')   # 이미 학습된 상태
lfw_model.summary()

# 예측
lfw_model.predict(lfw_model.images[:1])   # 이미지 하나만 예측해보자
    # cf) images[:1]이라고 해야 images의 shape이 그대로 전달됨 (images[0]는 X)

### 실습: 연예인 사진 분류

- 연예인 사진 저장 (남여 5장씩)
- 파일 형식 통일할 것 (jpg면 jpg, png면 png)
- 저장명:man_01, woman_01, woman_05

In [ ]:
import glob                # 특정 확장자나 이름 패턴을 가진 파일들을 자동으로 탐색해줌
from PIL import Image      # 파이썬에서 이미지를 다룰 때 사용하는 모듈


x_data = []    # 독립변수 담을 빈 리스트
y_data = []    # 타깃변수 담을 빈 리스트


for imgname in glob.glob(rootdir + 'celeb/*.jpg'):    # 해당 경로 내 .jpg 파일 전부 끌어와 하나씩 순회

  print(imgname)                    # 이미지 파일 경로 출력

  img = Image.open(imgname)         # 이미지 파일 열기
  img = img.resize((100,100))       # 이미지 크기 통일

  ndimg = np.array(img)             # 이미지 객체를 numpy 배열로 변환
                                    # (이미지의 픽셀값을 숫자화 하여 모델이 이해할 수 있도록 하는 것)

  x_data.append(ndimg[:,:,:3])      # 세로, 가로, 채널에서 채

  print(ndimg.shape)
  plt.imshow(ndimg)
  plt.show()

x_data = np.array(x_data)
y_data = np.array(y_data)
print(x_data.shape, y_data.shape)

In [ ]:
def celeb_predict(pth):
    ret = '남자 연예인입니다.'
    try:
        img = Image.open(pth).resize((100,100))
    except:
        return '파일에 문제가 있습니다.'
    
    ndimg = np.array(img) / 255
    ndimg = ndimg[:,:,:3]
    plt.imshow(ndimg)
    plt.show()
    try:
        celeb_model = load_model(rootdir + 'celeb_model.keras')
    except:
        return '딥러닝 모델 로드에 문제가 있습니다.'
    
    pred = celeb_model.predict(ndimg[np.newaxis,:,:,:]).squeeze()
    if pred > 0.5:
        ret = '여자 연예인입니다.'
    return ret